In [ ]:
!pip install smplx trimesh yacs chumpy -q
!sed -i "s/order is 'C'/order == 'C'/g" /usr/local/lib/python3.12/dist-packages/chumpy/ch_ops.py
!git clone --recurse-submodules -j4 https://github.com/filby89/spectre -q
!pip install pyrender moviepy imageio[ffmpeg] -q
!apt-get install -y libegl1-mesa libegl1-mesa-dev -qq

In [ ]:
import os
import sys
import cv2
import time
import glob
import math
import smplx
import torch
import shutil
import random
import imageio
import trimesh
import inspect
import pyrender
import numpy as np
from tqdm import tqdm
from enum import Enum
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.rnn as rnn_utils
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler, random_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from moviepy.editor import VideoFileClip, AudioFileClip
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

sys.path.append('/kaggle/working/spectre')
from config import cfg as spectre_cfg
from src.models.FLAME import FLAME

np.bool = np.bool_
np.int = np.int64
np.float = np.float64
np.complex = np.complex128
np.object = np.object_
np.unicode = np.str_
np.str = np.str_

In [ ]:
class OutputMode(Enum):
    FLAME = "flame"
    LATENT = "latent"

class DenoiserType(Enum):
    PLAIN = "plain"
    GRU_UNI = "gru_uni"
    GRU_BI = "gru_bi"
    TRANSFORMER = "transformer"

class TrainingMode(Enum):
    NORMAL = "normal"
    OVERFIT_TEST = "overfit_test"

OUTPUT_MODE = OutputMode.LATENT
DENOISER = DenoiserType.GRU_BI
ACTIVE_MODE = TrainingMode.NORMAL

BATCH_SIZE = 16
LR = 5e-5
WEIGHT_DECAY = 1e-4
MAX_TIME_HOURS = 11.5
EMA_DECAY = 0.9999
EMA_START_EPOCH = 5
TRAIN_TIMESTEPS = 1000
INFER_TIMESTEPS = 50
SEED = 0

if OUTPUT_MODE == OutputMode.FLAME:
    EPOCHS = 250 if DENOISER == DenoiserType.TRANSFORMER else 300
    EMA_START_EPOCH = 20
else:
    EPOCHS = 200 if DENOISER == DenoiserType.GRU_BI else 150
    EMA_START_EPOCH = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
flame_model_path = '/kaggle/input/datasets/jorx04/flame-2020-weights/FLAME2020/generic_model.pkl'
spectre_cfg.defrost()
spectre_cfg.model.flame_model_path = flame_model_path
spectre_cfg.model.use_tex = False
spectre_cfg.freeze()

flame_model = FLAME(spectre_cfg.model).to(device)

def fix_module_tensors(module):
    for attr_name, attr_value in list(vars(module).items()):
        if isinstance(attr_value, torch.Tensor) and not isinstance(attr_value, nn.Parameter):
            delattr(module, attr_name)
            module.register_buffer(attr_name, attr_value)
    for child in module.children():
        fix_module_tensors(child)

fix_module_tensors(flame_model)
flame_model.eval()

In [ ]:
def augment_mhubert(audio_feat, time_mask_max=10, p_channel_drop=0.4):
    aug_feat = audio_feat.clone()
    
    if torch.rand(1).item() < 0.5:
        T = aug_feat.shape[1]
        mask_len = torch.randint(1, time_mask_max + 1, (1,)).item()
        if T > mask_len:
            start = torch.randint(0, T - mask_len + 1, (1,)).item()
            aug_feat[:, start:start+mask_len, :] = 0.0 
            
    if torch.rand(1).item() < 0.5:
        drop_mask = torch.rand(768) < p_channel_drop
        aug_feat[:, :, drop_mask] = 0.0
        
    return aug_feat

In [ ]:
class FullSequenceAudio2FaceDataset(Dataset):
    def __init__(self, hubert_dirs, target_dirs, mode=TrainingMode.NORMAL, output_mode=OutputMode.FLAME):
        self.mode = mode
        self.output_mode = output_mode
        self.samples = []
        self.max_len = 75

        for h_dir, t_dir in zip(hubert_dirs, target_dirs):
            target_files = glob.glob(os.path.join(t_dir, "*.pt"))
            for t_path in target_files:
                base_name = os.path.splitext(os.path.basename(t_path))[0]
                clean_name = base_name.replace("_flame_params", "")
                h_path = os.path.join(h_dir, f"{clean_name}.pt") 
                if os.path.exists(h_path):
                    self.samples.append((h_path, t_path))
                    
        if mode == TrainingMode.OVERFIT_TEST and len(self.samples) > 0:
            self.samples = [self.samples[415]]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        h_path, t_path = self.samples[idx]
        audio_feat = torch.load(h_path, weights_only=True)
        
        target_data = torch.load(t_path, weights_only=True)
        shape_param = torch.zeros(1, 100)
        
        if self.output_mode == OutputMode.FLAME:
            expressions = target_data['exp'] 
            jaw_pose = target_data['pose'][:, 3:6]
            shape_param = target_data['shape'][0:1]
            target_seq = torch.cat([expressions, jaw_pose], dim=-1)[:, :53]
            stride = 2
        else:
            target_seq = target_data['latent']
            if 'shape' in target_data:
                shape_param = target_data['shape']
            stride = 4

        valid_len = min(target_seq.shape[0], audio_feat.shape[1] // stride)
        
        if self.max_len is not None and valid_len > self.max_len:
            start_idx = torch.randint(0, valid_len - self.max_len + 1, (1,)).item() if self.mode == TrainingMode.NORMAL else 0
            valid_len = self.max_len
        else:
            start_idx = 0

        t_seq = target_seq[start_idx : start_idx + valid_len]
        a_seq = audio_feat[:, start_idx * stride : (start_idx + valid_len) * stride]

        if self.mode == TrainingMode.NORMAL:
            a_seq = augment_mhubert(a_seq)
            
        return a_seq, t_seq, shape_param

In [ ]:
def pad_collate_fn(batch):
    audios = [item[0].transpose(0, 1) for item in batch]
    targets = [item[1] for item in batch]
    shapes = [item[2] for item in batch]
    
    lengths = torch.tensor([f.shape[0] for f in targets], dtype=torch.long)
    max_len = lengths.max()
    mask = torch.arange(max_len).expand(len(lengths), max_len) < lengths.unsqueeze(1)
    
    audios_padded = rnn_utils.pad_sequence(audios, batch_first=True).transpose(1, 2)
    targets_padded = rnn_utils.pad_sequence(targets, batch_first=True)
    shapes_batched = torch.stack(shapes, dim=0)
    
    return {
        "audio": audios_padded,
        "target": targets_padded,
        "shape": shapes_batched,
        "mask": mask
    }

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        return torch.cat((freqs, freqs), dim=-1)

def apply_rotary_emb(x, freqs):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    x_rot = torch.cat((-x2, x1), dim=-1)
    return x * freqs.cos() + x_rot * freqs.sin()

class SDPATransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim_feedforward, d_model)
        )
        self.nhead = nhead

    def forward(self, x, rope_freqs, mask=None):
        B, T, C = x.shape
        H = self.nhead
        D = C // H
        
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)
        q, k, v = q.view(B, T, H, D), k.view(B, T, H, D), v.view(B, T, H, D)
        
        q = apply_rotary_emb(q, rope_freqs.unsqueeze(0).unsqueeze(2))
        k = apply_rotary_emb(k, rope_freqs.unsqueeze(0).unsqueeze(2))
        
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        
        attn_mask = mask.unsqueeze(1).unsqueeze(2) if mask is not None else None
        
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0 if not self.training else 0.1)
        x = x + self.proj(out.transpose(1, 2).reshape(B, T, C))
        x = x + self.ffn(self.norm2(x))
        return x

class FLAMEKLEncoder(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.compression = compression
        self.input_proj  = nn.Linear(flame_dim, hidden_dim)
        self.rope = RotaryEmbedding(hidden_dim // num_heads)
        self.layers = nn.ModuleList([SDPATransformerLayer(hidden_dim, num_heads, hidden_dim * 4, dropout) for _ in range(num_layers)])
        self.temporal_pool = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=2, stride=2)
        self.to_mean    = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar  = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, mask=None):
        h = self.input_proj(x)
        freqs = self.rope(h.shape[1], h.device)
        for layer in self.layers:
            h = layer(h, freqs, mask)
        if self.compression > 1:
            h = self.temporal_pool(h.transpose(1, 2)).transpose(1, 2)
        return self.to_mean(h), self.to_logvar(h)

class FLAMEKLDecoder(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.compression = compression
        self.input_proj  = nn.Linear(latent_dim, hidden_dim)
        self.temporal_upsample = nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=2, stride=2)
        self.rope = RotaryEmbedding(hidden_dim // num_heads)
        self.layers = nn.ModuleList([SDPATransformerLayer(hidden_dim, num_heads, hidden_dim * 4, dropout) for _ in range(num_layers)])
        self.out_proj    = nn.Linear(hidden_dim, flame_dim)

    def forward(self, z, mask=None):
        h = self.input_proj(z)
        if self.compression > 1:
            h = self.temporal_upsample(h.transpose(1, 2)).transpose(1, 2)
        
        freqs = self.rope(h.shape[1], h.device)
        aligned_mask = mask[:, :h.shape[1]] if mask is not None else None
        
        for layer in self.layers:
            h = layer(h, freqs, aligned_mask)
        return self.out_proj(h)

class FLAMEKLVAE(nn.Module):
    def __init__(self, flame_dim=53, hidden_dim=256, latent_dim=32, num_layers=4, num_heads=4, dropout=0.1, compression=2):
        super().__init__()
        self.encoder = FLAMEKLEncoder(flame_dim, hidden_dim, latent_dim, num_layers, num_heads, dropout, compression)
        self.decoder = FLAMEKLDecoder(flame_dim, hidden_dim, latent_dim, num_layers, num_heads, dropout, compression)
        self.latent_dim  = latent_dim
        self.compression = compression

    def reparameterise(self, mean, logvar):
        if self.training:
            std = (0.5 * logvar).exp()
            return mean + std * torch.randn_like(std)
        return mean

    def encode(self, x, mask=None):
        mean, _ = self.encoder(x, mask)
        return mean

    def forward(self, x, mask=None):
        mean, logvar = self.encoder(x, mask)
        logvar = logvar.clamp(-30.0, 20.0)
        z = self.reparameterise(mean, logvar)
        recon = self.decoder(z, mask)
        recon = recon[:, :x.shape[1], :]
        return recon, mean, logvar

INPUT_FLAME_BASE = "/kaggle/input/datasets/jorx04/fully-collected-dataset/flame"
CUSTOM_INPUT_LATENT_BASE = None

vae_model = None
if OUTPUT_MODE == OutputMode.LATENT:
    vae_model = FLAMEKLVAE().to(device)
    
    vae_ckpt = torch.load("/kaggle/input/notebooks/jousefna/kl-vae-v2/klvae_ep100.pth", map_location=device)
    vae_model.load_state_dict(vae_ckpt['klvae_state'])
    vae_model.eval()
    for param in vae_model.parameters():
        param.requires_grad = False
        
    for lang in ['ara', 'eng']:
        os.makedirs(f"/kaggle/working/latents/{lang}", exist_ok=True)
        flame_files = glob.glob(os.path.join(INPUT_FLAME_BASE, lang, "*.pt"))
        
        for f_path in tqdm(flame_files):
            file_name = os.path.basename(f_path).replace('_flame_params', '')
            out_path = f"/kaggle/working/latents/{lang}/{file_name}"
            custom_in_path = None
            if CUSTOM_INPUT_LATENT_BASE:
                custom_in_path = os.path.join(CUSTOM_INPUT_LATENT_BASE, lang, file_name)
            
            if os.path.exists(out_path) or (custom_in_path and os.path.exists(custom_in_path)):
                continue
                
            flame_dict = torch.load(f_path, weights_only=True)
            flame_seq = torch.cat([flame_dict['exp'], flame_dict['pose'][:, 3:6]], dim=-1)[:, :53].unsqueeze(0).to(device)
            
            latent = vae_model.encode(flame_seq).squeeze(0).cpu()
            
            torch.save({'latent': latent, 'shape': flame_dict['shape'][0:1]}, out_path)

In [ ]:
class LightweightAudioStrider(nn.Module):
    def __init__(self, output_dim=512, num_heads=4, output_mode="flame"):
        super().__init__()
        self.stride = 2 if output_mode == "flame" else 4
        self.layer_weights = nn.Parameter(torch.ones(12))
        self.temporal_stride = nn.Conv1d(in_channels=768, out_channels=output_dim, kernel_size=3, stride=self.stride, padding=1)
        self.activation = nn.GELU()
        self.self_attention = nn.MultiheadAttention(embed_dim=output_dim, num_heads=num_heads, batch_first=True)

    def forward(self, precomputed_audio):
        norm_weights = F.softmax(self.layer_weights, dim=0).view(1, 12, 1, 1)
        weighted_sum = (precomputed_audio * norm_weights).sum(dim=1).transpose(1, 2)
        strided_features = self.activation(self.temporal_stride(weighted_sum)).transpose(1, 2)
        attn_out, _ = self.self_attention(strided_features, strided_features, strided_features)
        return strided_features + attn_out

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

In [ ]:
def build_time_mlp():
    return nn.Sequential(
        SinusoidalPositionEmbeddings(128),
        nn.Linear(128, 256), nn.GELU(), nn.Linear(256, 256)
    )

class PlainDenoiser(nn.Module):
    def __init__(self, out_dim, hidden_dim=512):
        super().__init__()
        self.time_mlp = build_time_mlp()
        self.input_proj = nn.Linear(out_dim, hidden_dim)
        layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, dim_feedforward=2048, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=4)
        self.output_proj = nn.Linear(hidden_dim, out_dim)

    def forward(self, xt, audio, t, hidden_state=None):
        t_emb = self.time_mlp(t).unsqueeze(1)
        h = self.input_proj(xt) + t_emb
        h = h + audio
        h = self.transformer(h)
        return self.output_proj(h), None

class GRUUniDenoiser(nn.Module):
    def __init__(self, out_dim, hidden_dim=512):
        super().__init__()
        self.time_mlp = build_time_mlp()
        self.xt_proj = nn.Sequential(nn.Linear(out_dim, 256), nn.GELU())
        self.gru = nn.GRU(input_size=1024, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.4)
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.output_proj = nn.Sequential(nn.Linear(hidden_dim, 256), nn.GELU(), nn.Linear(256, out_dim))

    def forward(self, xt, audio, t, hidden_state=None):
        t_emb = self.time_mlp(t).unsqueeze(1).expand(-1, xt.shape[1], -1)
        xt_feat = self.xt_proj(xt)
        fused = torch.cat([xt_feat, audio, t_emb], dim=-1)
        gru_out, new_hidden = self.gru(fused, hidden_state)
        
        attn_out, _ = self.attn(gru_out, gru_out, gru_out)
        out = self.norm(gru_out + attn_out)
        return self.output_proj(out), new_hidden

class GRUBiDenoiser(nn.Module):
    def __init__(self, out_dim, hidden_dim=512):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(128),
            nn.Linear(128, 256), nn.GELU(), nn.Linear(256, 512)
        )
        self.input_proj = nn.Linear(out_dim, hidden_dim)
        
        self.self_attn1 = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)
        self.cross_attn1 = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.self_attn2 = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)
        self.cross_attn2 = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)
        
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(5)])
        self.output_proj = nn.Linear(hidden_dim, out_dim)

    def forward(self, xt, audio, t, hidden_state=None):
        h = self.input_proj(xt) + self.time_mlp(t).unsqueeze(1)
        
        sa1, _ = self.self_attn1(h, h, h)
        h = self.norms[0](h + sa1)
        ca1, _ = self.cross_attn1(h, audio, audio)
        h = self.norms[1](h + ca1)
        
        gru_out, _ = self.gru(h)
        h = self.norms[2](h + gru_out)
        
        sa2, _ = self.self_attn2(h, h, h)
        h = self.norms[3](h + sa2)
        ca2, _ = self.cross_attn2(h, audio, audio)
        h = self.norms[4](h + ca2)
        
        return self.output_proj(h), None

class SDPA_CrossAttentionLayer(nn.Module):
    def __init__(self, d_model, nhead):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.nhead = nhead

    def forward(self, x, context):
        B, T, C = x.shape
        T_ctx = context.shape[1]
        H = self.nhead
        D = C // H
        
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k, v = self.kv_proj(context).chunk(2, dim=-1)
        k = k.view(B, T_ctx, H, D).transpose(1, 2)
        v = v.view(B, T_ctx, H, D).transpose(1, 2)
        
        out = F.scaled_dot_product_attention(q, k, v)
        return self.proj(out.transpose(1, 2).reshape(B, T, C))

class TransformerDenoiser(nn.Module):
    def __init__(self, out_dim, hidden_dim=512):
        super().__init__()
        self.time_mlp = build_time_mlp()
        self.input_proj = nn.Linear(out_dim, hidden_dim)
        self.rope = RotaryEmbedding(hidden_dim // 8)
        
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'self_attn': SDPATransformerLayer(hidden_dim, 8, 2048, 0.1),
                'cross_attn': SDPA_CrossAttentionLayer(hidden_dim, 8),
                'norm_ca': nn.LayerNorm(hidden_dim)
            }) for _ in range(6)
        ])
        self.output_proj = nn.Linear(hidden_dim, out_dim)

    def forward(self, xt, audio, t, hidden_state=None):
        h = self.input_proj(xt) + self.time_mlp(t).unsqueeze(1)
        freqs = self.rope(h.shape[1], h.device)
        
        for layer in self.layers:
            h = layer['self_attn'](h, freqs)
            ca_out = layer['cross_attn'](layer['norm_ca'](h), audio)
            h = h + ca_out
            
        return self.output_proj(h), None

In [ ]:
class FlowMatchingDiffusion(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, gt_x0, strided_audio):
        B, T, D = gt_x0.shape
        t = torch.rand(B, 1, 1, device=gt_x0.device)
        noise = torch.randn_like(gt_x0)
        
        x_t = (1 - t) * noise + t * gt_x0
        pred_vt, _ = self.model(x_t, strided_audio, t.squeeze())
        pred_x0 = x_t + (1 - t) * pred_vt 
        
        return pred_x0

In [ ]:
raw_mouth = [1576, 1577, 1578, 1579, 1583, 1657, 1667, 1668, 1669, 1670, 1691, 1693, 1694, 1695, 1696, 1697, 1700, 1702, 1703, 1704, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1716, 1717, 1719, 1720, 1721, 1731, 1732, 1733, 1734, 1735, 1736, 1737, 1738, 1740, 1748, 1749, 1750, 1751, 1758, 1763, 1770, 1771, 1773, 1774, 1775, 1776, 1777, 1778, 1787, 1788, 1789, 1791, 1792, 1793, 1794, 1795, 1796, 1801, 1802, 1803, 1804, 1826, 1827, 1836, 1848, 1849, 1850, 1865, 1866, 2712, 2713, 2714, 2715, 2716, 2719, 2774, 2784, 2785, 2786, 2787, 2811, 2812, 2813, 2814, 2817, 2819, 2820, 2821, 2826, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2836, 2837, 2838, 2840, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2855, 2863, 2864, 2865, 2866, 2869, 2871, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2890, 2891, 2892, 2894, 2895, 2896, 2897, 2898, 2899, 2904, 2905, 2906, 2907, 2928, 2929, 2934, 2937, 2938, 2939, 2948, 2949, 3503, 3504, 3506, 3509, 3511, 3513, 3531, 3533, 3543, 3546, 3790, 3791, 3792, 3793, 3795, 3796, 3797, 3798, 3799, 3800, 3805, 3806, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3927, 3928]
raw_lower = [63, 64, 123, 124, 143, 174, 177, 183, 184, 223, 224, 245, 250, 270, 271, 365, 532, 560, 561, 562, 565, 566, 567, 568, 613, 726, 729, 784, 785, 856, 859, 867, 868, 933, 934, 1009, 1049, 1050, 1087, 1092, 1479, 1480, 1496, 1530, 1531, 1569, 1570, 1574, 1575, 1576, 1577, 1578, 1579, 1584, 1585, 1586, 1587, 1596, 1597, 1598, 1599, 1601, 1602, 1657, 1658, 1669, 1670, 1686, 1687, 1688, 1689, 1690, 1691, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1716, 1717, 1731, 1732, 1733, 1734, 1735, 1736, 1737, 1738, 1749, 1750, 1751, 1758, 1763, 1766, 1767, 1768, 1769, 1770, 1771, 1773, 1774, 1775, 1776, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1826, 1848, 1849, 1850, 1853, 1863, 1864, 1865, 1866, 1935, 1936, 1937, 1960, 1961, 1962, 1963, 1998, 1999, 2004, 2009, 2011, 2012, 2078, 2079, 2080, 2081, 2083, 2175, 2185, 2208, 2211, 2252, 2399, 2401, 2616, 2617, 2666, 2667, 2705, 2706, 2710, 2711, 2712, 2713, 2714, 2715, 2720, 2721, 2722, 2723, 2732, 2733, 2734, 2735, 2737, 2738, 2774, 2775, 2786, 2787, 2803, 2804, 2805, 2806, 2807, 2808, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2864, 2865, 2866, 2871, 2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 2883, 2890, 2891, 2894, 2895, 2896, 2897, 2898, 2899, 2900, 2901, 2902, 2903, 2904, 2905, 2906, 2907, 2928, 2937, 2938, 2939, 2946, 2947, 2948, 2949, 3054, 3055, 3056, 3057, 3059, 3060, 3112, 3113, 3114, 3115, 3116, 3118, 3127, 3183, 3381, 3382, 3383, 3384, 3385, 3386, 3387, 3388, 3389, 3390, 3391, 3392, 3393, 3394, 3395, 3396, 3397, 3398, 3399, 3400, 3401, 3402, 3403, 3404, 3405, 3406, 3407, 3408, 3409, 3410, 3411, 3412, 3413, 3414, 3415, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3424, 3425, 3426, 3427, 3428, 3429, 3430, 3431, 3432, 3433, 3434, 3435, 3436, 3437, 3438, 3439, 3440, 3441, 3442, 3443, 3444, 3445, 3446, 3447, 3448, 3449, 3450, 3451, 3453, 3454, 3455, 3456, 3457, 3458, 3459, 3464, 3465, 3466, 3467, 3468, 3469, 3470, 3471, 3472, 3473, 3474, 3475, 3476, 3477, 3478, 3479, 3480, 3481, 3482, 3483, 3484, 3485, 3486, 3487, 3489, 3490, 3491, 3492, 3493, 3499, 3502, 3503, 3504, 3506, 3509, 3511, 3515, 3531, 3537, 3538, 3541, 3543, 3546, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3587, 3588, 3593, 3594, 3595, 3596, 3598, 3599, 3600, 3601, 3604, 3605, 3611, 3614, 3623, 3624, 3625, 3626, 3628, 3629, 3630, 3634, 3635, 3636, 3637, 3643, 3644, 3646, 3649, 3650, 3652, 3653, 3654, 3655, 3656, 3658, 3659, 3660, 3662, 3663, 3664, 3665, 3666, 3667, 3670, 3671, 3672, 3673, 3676, 3677, 3678, 3679, 3680, 3681, 3685, 3691, 3693, 3697, 3698, 3701, 3703, 3707, 3714, 3715, 3716, 3717, 3722, 3724, 3725, 3726, 3727, 3728, 3730, 3734, 3737, 3738, 3739, 3740, 3742, 3745, 3752, 3753, 3754, 3756, 3757, 3761, 3762, 3769, 3771, 3772, 3790, 3791, 3792, 3793, 3794, 3795, 3796, 3797, 3798, 3799, 3800, 3801, 3802, 3803, 3804, 3805, 3806, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3924, 3925, 3926, 3927, 3928]
raw_upper = [16, 17, 18, 27, 335, 336, 337, 338, 498, 499, 500, 501, 569, 570, 571, 572, 589, 590, 591, 592, 605, 622, 623, 624, 625, 626, 627, 628, 629, 630, 667, 668, 669, 670, 671, 672, 673, 674, 679, 680, 681, 682, 683, 688, 693, 694, 695, 696, 697, 702, 703, 704, 705, 706, 707, 708, 713, 714, 715, 723, 724, 725, 738, 739, 807, 808, 809, 814, 815, 816, 821, 822, 823, 824, 825, 827, 828, 829, 841, 842, 848, 864, 865, 877, 878, 879, 880, 881, 882, 883, 884, 885, 896, 897, 898, 899, 902, 903, 904, 905, 906, 907, 908, 909, 918, 919, 922, 923, 924, 926, 927, 928, 929, 939, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 977, 978, 979, 980, 985, 986, 991, 994, 995, 999, 1012, 1013, 1014, 1015, 1019, 1020, 1021, 1022, 1023, 1033, 1034, 1043, 1044, 1059, 1060, 1062, 1088, 1093, 1096, 1101, 1108, 1113, 1114, 1115, 1125, 1132, 1134, 1135, 1142, 1143, 1144, 1146, 1147, 1151, 1152, 1153, 1154, 1155, 1161, 1162, 1163, 1164, 1168, 1169, 1170, 1175, 1176, 1181, 1182, 1183, 1184, 1189, 1190, 1193, 1194, 1195, 1200, 1201, 1202, 1216, 1218, 1225, 1226, 1232, 1233, 1243, 1244, 1292, 1293, 1294, 1329, 1331, 1336, 1337, 1338, 1339, 1340, 1341, 1342, 1343, 1344, 1345, 1346, 1347, 1348, 1349, 1350, 1351, 1352, 1353, 1354, 1355, 1356, 1357, 1358, 1361, 1362, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1370, 1371, 1372, 1373, 1374, 1375, 1376, 1377, 1378, 1383, 1384, 1385, 1387, 1388, 1389, 1390, 1391, 1396, 1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1425, 1426, 1427, 1428, 1429, 1430, 1431, 1432, 1433, 1434, 1435, 1436, 1437, 1438, 1439, 1440, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451, 1452, 1453, 1458, 1459, 1460, 1461, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1473, 1474, 1475, 1476, 1477, 1478, 1641, 1642, 1643, 1644, 1649, 1752, 1753, 1754, 1755, 1982, 1983, 1984, 1985, 2017, 2018, 2019, 2020, 2030, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2053, 2054, 2058, 2059, 2060, 2061, 2062, 2063, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2082, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2134, 2135, 2138, 2139, 2140, 2165, 2166, 2167, 2168, 2176, 2177, 2178, 2179, 2180, 2181, 2182, 2183, 2184, 2185, 2188, 2189, 2190, 2191, 2192, 2193, 2194, 2195, 2196, 2197, 2198, 2199, 2202, 2203, 2204, 2205, 2206, 2207, 2220, 2221, 2264, 2265, 2266, 2267, 2268, 2269, 2270, 2271, 2272, 2273, 2274, 2276, 2277, 2278, 2282, 2283, 2286, 2287, 2288, 2289, 2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300, 2301, 2302, 2303, 2304, 2305, 2306, 2307, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316, 2317, 2318, 2319, 2320, 2321, 2322, 2323, 2324, 2325, 2326, 2327, 2328, 2329, 2330, 2331, 2332, 2333, 2334, 2335, 2336, 2337, 2338, 2339, 2340, 2341, 2342, 2343, 2344, 2345, 2346, 2347, 2348, 2349, 2350, 2351, 2352, 2353, 2354, 2358, 2359, 2360, 2370, 2371, 2372, 2373, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2388, 2389, 2391, 2400, 2402, 2403, 2404, 2405, 2406, 2407, 2408, 2411, 2416, 2417, 2418, 2419, 2420, 2421, 2422, 2423, 2425, 2426, 2427, 2428, 2429, 2430, 2431, 2432, 2433, 2434, 2435, 2436, 2437, 2438, 2439, 2440, 2441, 2442, 2443, 2444, 2445, 2446, 2447, 2448, 2449, 2450, 2451, 2452, 2453, 2454, 2455, 2456, 2461, 2462, 2465, 2466, 2471, 2472, 2473, 2485, 2486, 2487, 2488, 2489, 2490, 2491, 2492, 2493, 2494, 2495, 2496, 2497, 2498, 2499, 2500, 2501, 2502, 2503, 2504, 2505, 2506, 2507, 2508, 2509, 2510, 2511, 2512, 2513, 2514, 2515, 2516, 2517, 2518, 2519, 2520, 2521, 2522, 2523, 2524, 2525, 2526, 2527, 2528, 2529, 2530, 2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539, 2540, 2541, 2542, 2543, 2544, 2545, 2546, 2547, 2548, 2549, 2550, 2551, 2552, 2553, 2554, 2555, 2556, 2557, 2558, 2559, 2562, 2563, 2564, 2565, 2566, 2567, 2568, 2569, 2570, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2579, 2580, 2581, 2582, 2583, 2584, 2585, 2586, 2587, 2588, 2589, 2590, 2595, 2596, 2597, 2598, 2599, 2600, 2601, 2602, 2603, 2604, 2605, 2606, 2610, 2611, 2612, 2613, 2614, 2615, 2741, 2742, 2758, 2759, 2760, 2761, 2766, 3068, 3078, 3079, 3080, 3081, 3082, 3083, 3084, 3085, 3086, 3088, 3089, 3090, 3093, 3094, 3095, 3096, 3097, 3098, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3117, 3119, 3120, 3121, 3122, 3123, 3124, 3125, 3126, 3127, 3128, 3129, 3130, 3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3141, 3145, 3146, 3147, 3148, 3149, 3150, 3151, 3152, 3153, 3154, 3157, 3158, 3159, 3495, 3516, 3518, 3524, 3540, 3542, 3550, 3553, 3555, 3560, 3572, 3589, 3592, 3603, 3615, 3619, 3620, 3622, 3631, 3632, 3638, 3645, 3647, 3648, 3674, 3675, 3682, 3683, 3684, 3686, 3687, 3688, 3689, 3690, 3692, 3694, 3696, 3699, 3704, 3705, 3706, 3708, 3710, 3711, 3712, 3718, 3720, 3721, 3729, 3731, 3732, 3733, 3735, 3741, 3743, 3744, 3747, 3749, 3750, 3759, 3763, 3764, 3766, 3767, 3770, 3773, 3774, 3775, 3776, 3777, 3779, 3780, 3781, 3784, 3786, 3787, 3788, 3789, 3809, 3812, 3815, 3823, 3827, 3828, 3830, 3832, 3833, 3835, 3840, 3841, 3842, 3847, 3848, 3849, 3850, 3851, 3852, 3854, 3855, 3856, 3857, 3858, 3859, 3860, 3863, 3864, 3865, 3866, 3867, 3868, 3869, 3871, 3872, 3874, 3875, 3876, 3877, 3878, 3880, 3881, 3882, 3884, 3886, 3887, 3891, 3892, 3893, 3895, 3896, 3898, 3899, 3900, 3901, 3902, 3903, 3905, 3906, 3907, 3910, 3911, 3912, 3913, 3965, 4021, 4041, 4271, 4556, 4563, 4609]

mouth_set = set(raw_mouth)
lower_set = set(raw_lower) - mouth_set
upper_set = set(raw_upper) - mouth_set - lower_set

clean_mouth = sorted(list(mouth_set))
clean_lower = sorted(list(lower_set))
clean_upper = sorted(list(upper_set))

In [ ]:
class DynamicLossWeights:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs

    def get_weights(self, epoch, output_mode):
        prog = epoch / self.total_epochs
        weights = {}

        if output_mode == OutputMode.FLAME:
            weights['w_recon'] = 5.0 - 3.0 * prog
            weights['w_motion'] = 0.0 if epoch < 30 else 0.05 + 0.75 * min((epoch - 30) / (self.total_epochs - 30), 1.0)
            weights['w_jaw'] = 10.0 + 140.0 * prog
            
            base_mouth = 1000.0 + 2000.0 * prog
            base_lower = 500.0 + 1000.0 * prog
            
            cycle_prog = (epoch % 100) / 100.0
            dip = 0.6 if cycle_prog < 0.15 else 1.0
            
            weights['w_mouth'] = base_mouth * dip
            weights['w_lower'] = base_lower * dip
            weights['w_upper'] = 2000.0 - 1500.0 * prog
            
        else:
            weights['w_latent_mse'] = 1.0
            
            w_v = 0.0 if epoch < 30 else 0.15 * min((epoch - 30) / (self.total_epochs - 30), 1.0)
            cycle_prog = (epoch % 100) / 100.0
            dip = 0.5 if cycle_prog < 0.15 else 1.0
            weights['w_vertex'] = w_v * dip
            
            weights['w_jaw'] = 10.0 + 70.0 * prog
            weights['w_mouth'] = 500.0 + 1000.0 * prog
            weights['w_lower'] = 250.0 + 500.0 * prog
            weights['w_upper'] = 1000.0 - 700.0 * prog
            weights['w_motion'] = 0.0 if epoch < 30 else 0.5 * min((epoch - 30) / (self.total_epochs - 30), 1.0)
            weights['w_recon'] = 1.0
            
        return weights

class VertexAudio2FaceLoss(nn.Module):
    def __init__(self, flame_model, device, output_mode=OutputMode.FLAME, vae_decoder=None):
        super().__init__()
        self.flame = flame_model
        self.device = device
        self.output_mode = output_mode
        self.vae_decoder = vae_decoder

        self.register_buffer('mouth_indices', torch.tensor(clean_mouth, dtype=torch.long))
        self.register_buffer('lower_indices', torch.tensor(clean_lower, dtype=torch.long))
        self.register_buffer('upper_indices', torch.tensor(clean_upper, dtype=torch.long))
        self.weights = {}

    def update_weights(self, weights):
        self.weights = weights

    def _get_vertices(self, params_flat, shape_expanded):
        B_T = params_flat.shape[0]
        exp = params_flat[:, :50]
        pose = torch.zeros(B_T, 6, device=self.device)
        pose[:, 3:6] = params_flat[:, 50:53]
        return self.flame(shape_params=shape_expanded, expression_params=exp, pose_params=pose)[0]

    def forward(self, pred, gt, mask, gt_shape):
        B, T, D = pred.shape
        w = self.weights
        
        loss_total = 0.0
        if self.output_mode == OutputMode.LATENT:
            latent_mse = F.mse_loss(pred, gt, reduction='none')
            m_expanded = mask.unsqueeze(-1).float()
            loss_total += w.get('w_latent_mse', 1.0) * (latent_mse * m_expanded).sum() / (m_expanded.sum() + 1e-8)
            
            if w.get('w_vertex', 0.0) == 0.0:
                return loss_total
            
            mask = mask.repeat_interleave(2, dim=1)
            
            with torch.no_grad():
                gt_flame = self.vae_decoder(gt, mask)
            pred_flame = self.vae_decoder(pred, mask)
            
            T = pred_flame.shape[1] 
            
        else:
            pred_flame = pred
            gt_flame = gt

        if gt_shape.dim() == 3: gt_shape = gt_shape.squeeze(1)
        shape_expanded = gt_shape.unsqueeze(1).expand(B, T, 100).reshape(B * T, 100)

        pred_v = self._get_vertices(pred_flame.reshape(B*T, 53), shape_expanded).reshape(B, T, 5023, 3)
        gt_v = self._get_vertices(gt_flame.reshape(B*T, 53), shape_expanded).reshape(B, T, 5023, 3)
        mask_v = mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred_v).float()
        sq_err = (pred_v - gt_v) ** 2

        spatial_weights = torch.ones(5023, device=self.device)
        spatial_weights[self.upper_indices] = w['w_upper']
        spatial_weights[self.lower_indices] = w['w_lower']
        spatial_weights[self.mouth_indices] = w['w_mouth']
        spatial_weights = spatial_weights.view(1, 1, 5023, 1)

        weighted_err = sq_err * spatial_weights * mask_v
        loss_recon = weighted_err.sum() / (mask_v.sum() + 1e-8)

        loss_motion = torch.tensor(0.0, device=self.device)
        if T > 2:
            accel = pred_v[:, 2:] - 2 * pred_v[:, 1:-1] + pred_v[:, :-2]
            motion_mask = mask_v[:, 2:]
            weighted_accel = (accel ** 2) / spatial_weights.unsqueeze(1)
            loss_motion = (weighted_accel * motion_mask).sum() / (motion_mask.sum() + 1e-8)

        jaw_err = (pred_flame[:, :, 50:53] - gt_flame[:, :, 50:53]) ** 2
        mask_jaw = mask.unsqueeze(-1).expand_as(jaw_err).float()
        loss_jaw = (jaw_err * mask_jaw).sum() / (mask_jaw.sum() + 1e-8)

        vertex_loss = (w['w_recon'] * loss_recon) + (w['w_motion'] * loss_motion) + (w['w_jaw'] * loss_jaw)
        
        if self.output_mode == OutputMode.LATENT:
            return loss_total + (w['w_vertex'] * vertex_loss)
        return vertex_loss

In [ ]:
target_dim = 53 if OUTPUT_MODE == OutputMode.FLAME else 32

audio_strider = LightweightAudioStrider(output_dim=512, num_heads=4, output_mode=OUTPUT_MODE.value).to(device)

if DENOISER == DenoiserType.PLAIN:
    denoiser_core = PlainDenoiser(out_dim=target_dim).to(device)
elif DENOISER == DenoiserType.GRU_UNI:
    denoiser_core = GRUUniDenoiser(out_dim=target_dim).to(device)
elif DENOISER == DenoiserType.GRU_BI:
    denoiser_core = GRUBiDenoiser(out_dim=target_dim).to(device)
elif DENOISER == DenoiserType.TRANSFORMER:
    denoiser_core = TransformerDenoiser(out_dim=target_dim).to(device)

diffusion_model = FlowMatchingDiffusion(denoiser_core).to(device)

decoder = vae_model.decoder if (OUTPUT_MODE == OutputMode.LATENT and vae_model) else None
criterion = VertexAudio2FaceLoss(flame_model, device, output_mode=OUTPUT_MODE, vae_decoder=decoder).to(device)
loss_scheduler = DynamicLossWeights(total_epochs=EPOCHS)

optimizer = torch.optim.AdamW(
    list(audio_strider.parameters()) + list(denoiser_core.parameters()), 
    lr=LR, weight_decay=0.01, fused=True
)

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=1, eta_min=1e-6)

ema_avg_fn = get_ema_multi_avg_fn(0.99)
ema_audio_strider = AveragedModel(audio_strider, multi_avg_fn=ema_avg_fn)
ema_denoiser = AveragedModel(denoiser_core, multi_avg_fn=ema_avg_fn)
ema_diffusion_model = FlowMatchingDiffusion(ema_denoiser).to(device)

RESUME_CHECKPOINT = "/kaggle/working/checkpoint_ep30.pth"
start_epoch = 0

if os.path.exists(RESUME_CHECKPOINT):
    print(f"Loading checkpoint from {RESUME_CHECKPOINT}")
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
    
    audio_strider.load_state_dict(checkpoint['audio_strider_state'])
    denoiser_core.load_state_dict(checkpoint['denoiser_state'])
    
    ema_audio_strider.module.load_state_dict(checkpoint['ema_audio_strider_state'])
    ema_denoiser.module.load_state_dict(checkpoint['ema_denoiser_state'])
    
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    
    start_epoch = checkpoint['epoch']
    
    if 'best_val_loss' in checkpoint:
        best_val_loss = checkpoint['best_val_loss']
        
    print(f"Successfully loaded checkpoint. Resuming at epoch {start_epoch}")
    
if torch.cuda.device_count() > 1:
    audio_strider = nn.DataParallel(audio_strider)
    denoiser_core = nn.DataParallel(denoiser_core)
    diffusion_model = nn.DataParallel(diffusion_model)
    criterion = nn.DataParallel(criterion)

In [ ]:
hubert_dirs = ["/kaggle/input/datasets/jorx04/mhubert-147-features-ara/ara", "/kaggle/input/datasets/jorx04/mhubert-147-features-eng/eng"]

if OUTPUT_MODE == OutputMode.FLAME:
    target_dirs = ["/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/ara", "/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/eng"]
else:
    target_dirs = ["/kaggle/working/latents/ara", "/kaggle/working/latents/eng"]

dataset = FullSequenceAudio2FaceDataset(hubert_dirs, target_dirs, mode=ACTIVE_MODE, output_mode=OUTPUT_MODE)

val_len = max(1, int(0.1 * len(dataset)))
train_len = len(dataset) - val_len
train_dataset, val_dataset = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))

train_weights = []
weight_eng = 1.0 / 11000.0
weight_ara = 1.0 / 8000.0
for idx in train_dataset.indices:
    _, t_path = dataset.samples[idx]
    if '/ara/' in t_path:
        train_weights.append(weight_ara)
    else:
        train_weights.append(weight_eng)

train_sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, 
    collate_fn=pad_collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)

val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    collate_fn=pad_collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)

scaler = torch.cuda.amp.GradScaler()

In [ ]:
max_time_seconds = 11 * 3600
start_time = time.time()
if 'best_val_loss' not in globals():
    best_val_loss = float('inf')

print("-" * 55)
print(f"| {'Epoch':^7} | {'Train Loss':^15} | {'Val Loss':^15} |")
print("-" * 55)

for epoch in range(start_epoch, EPOCHS):
    audio_strider.train()
    denoiser_core.train()
    
    train_dataloader.dataset.dataset.max_len = 75 if epoch < 50 else None
    cfg_prob = 0.10 if epoch < 50 else 0.20
    
    weights = loss_scheduler.get_weights(epoch, OUTPUT_MODE)
    base_crit = criterion.module if hasattr(criterion, 'module') else criterion
    base_crit.update_weights(weights)
    
    epoch_loss_sum = 0.0
    num_batches = 0
    
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        if time.time() - start_time > max_time_seconds:
            print("\n[Time Limit Reached] Stopping safely...")
            break
            
        audio = batch['audio'].to(device)
        target = batch['target'].to(device)
        mask = batch['mask'].to(device)
        shape_param = batch['shape'].to(device)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            strided_audio = audio_strider(audio)
            
            if random.random() < cfg_prob:
                strided_audio = torch.zeros_like(strided_audio)
                
            pred_x0 = diffusion_model(target, strided_audio)
            loss = criterion(pred_x0, target, mask, shape_param)
        
        scaler.scale(loss.mean()).backward() 
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(audio_strider.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(denoiser_core.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        if epoch >= EMA_START_EPOCH:
            base_audio_strider = audio_strider.module if hasattr(audio_strider, 'module') else audio_strider
            base_denoiser = denoiser_core.module if hasattr(denoiser_core, 'module') else denoiser_core
            ema_audio_strider.update_parameters(base_audio_strider)
            ema_denoiser.update_parameters(base_denoiser)
        
        epoch_loss_sum += loss.mean().item()
        num_batches += 1
        
    scheduler.step()
    avg_epoch_loss = epoch_loss_sum / max(num_batches, 1)

    if (epoch + 1) % 5 == 0 or (time.time() - start_time > max_time_seconds):
        model_to_eval = ema_diffusion_model if epoch >= EMA_START_EPOCH else diffusion_model
        strider_to_eval = ema_audio_strider if epoch >= EMA_START_EPOCH else audio_strider
        
        strider_to_eval.eval()
        model_to_eval.eval()
        
        val_loss_sum = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for val_batch in val_dataloader:
                val_audio = val_batch['audio'].to(device)
                val_target = val_batch['target'].to(device)
                val_mask = val_batch['mask'].to(device)
                val_shape = val_batch['shape'].to(device)
                
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    val_strided_audio = strider_to_eval(val_audio)
                    val_pred_x0 = model_to_eval(val_target, val_strided_audio) 
                    val_loss = criterion(val_pred_x0, val_target, val_mask, val_shape)
                
                val_loss_sum += val_loss.mean().item()
                val_batches += 1
                
        avg_val_loss = val_loss_sum / max(val_batches, 1)
        
        checkpoint_path = f"/kaggle/working/checkpoint_ep{epoch+1}.pth"
        base_deno = denoiser_core.module if hasattr(denoiser_core, 'module') else denoiser_core
        base_stride = audio_strider.module if hasattr(audio_strider, 'module') else audio_strider
        
        checkpoint = {
            'epoch': epoch + 1,
            'output_mode': OUTPUT_MODE.value,
            'denoiser_type': DENOISER.value,
            'audio_strider_state': base_stride.state_dict(),
            'denoiser_state': base_deno.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'ema_audio_strider_state': ema_audio_strider.module.state_dict() if hasattr(ema_audio_strider, 'module') else ema_audio_strider.state_dict(),
            'ema_denoiser_state': ema_denoiser.module.state_dict() if hasattr(ema_denoiser, 'module') else ema_denoiser.state_dict(),
            'best_val_loss': best_val_loss
        }
        
        torch.save(checkpoint, checkpoint_path)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(checkpoint, "/kaggle/working/best_model.pth")
            print(f"| {epoch + 1:^7} | {avg_epoch_loss:^15.4f} | {avg_val_loss:^15.4f} | --> Saved: {checkpoint_path} & BEST")
        else:
            print(f"| {epoch + 1:^7} | {avg_epoch_loss:^15.4f} | {avg_val_loss:^15.4f} | --> Saved: {checkpoint_path}")

    if time.time() - start_time > max_time_seconds:
        break

In [ ]:
max_time_seconds = 11 * 3600
start_time = time.time()
if 'best_val_loss' not in globals():
    best_val_loss = float('inf')

print("-" * 55)
print(f"| {'Epoch':^7} | {'Train Loss':^15} | {'Val Loss':^15} |")
print("-" * 55)

for epoch in range(start_epoch, EPOCHS):
    audio_strider.train()
    denoiser_core.train()
    
    train_dataloader.dataset.dataset.max_len = 75 if epoch < 50 else None
    cfg_prob = 0.10 if epoch < 50 else 0.20
    
    weights = loss_scheduler.get_weights(epoch, OUTPUT_MODE)
    base_crit = criterion.module if hasattr(criterion, 'module') else criterion
    base_crit.update_weights(weights)
    
    epoch_loss_sum = 0.0
    num_batches = 0
    
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        if time.time() - start_time > max_time_seconds:
            print("\n[Time Limit Reached] Stopping safely...")
            break
            
        audio = batch['audio'].to(device)
        target = batch['target'].to(device)
        mask = batch['mask'].to(device)
        shape_param = batch['shape'].to(device)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            strided_audio = audio_strider(audio)
            
            if random.random() < cfg_prob:
                strided_audio = torch.zeros_like(strided_audio)
                
            pred_x0 = diffusion_model(target, strided_audio)
            loss = criterion(pred_x0, target, mask, shape_param)
        
        scaler.scale(loss.mean()).backward() 
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(audio_strider.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(denoiser_core.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        if epoch >= EMA_START_EPOCH:
            base_audio_strider = audio_strider.module if hasattr(audio_strider, 'module') else audio_strider
            base_denoiser = denoiser_core.module if hasattr(denoiser_core, 'module') else denoiser_core
            ema_audio_strider.update_parameters(base_audio_strider)
            ema_denoiser.update_parameters(base_denoiser)
        
        epoch_loss_sum += loss.mean().item()
        num_batches += 1
        
    scheduler.step()
    avg_epoch_loss = epoch_loss_sum / max(num_batches, 1)

    if (epoch + 1) % 5 == 0 or (time.time() - start_time > max_time_seconds):
        model_to_eval = ema_diffusion_model if epoch >= EMA_START_EPOCH else diffusion_model
        strider_to_eval = ema_audio_strider if epoch >= EMA_START_EPOCH else audio_strider
        
        strider_to_eval.eval()
        model_to_eval.eval()
        
        val_loss_sum = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for val_batch in val_dataloader:
                val_audio = val_batch['audio'].to(device)
                val_target = val_batch['target'].to(device)
                val_mask = val_batch['mask'].to(device)
                val_shape = val_batch['shape'].to(device)
                
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    val_strided_audio = strider_to_eval(val_audio)
                    val_pred_x0 = model_to_eval(val_target, val_strided_audio) 
                    val_loss = criterion(val_pred_x0, val_target, val_mask, val_shape)
                
                val_loss_sum += val_loss.mean().item()
                val_batches += 1
                
        avg_val_loss = val_loss_sum / max(val_batches, 1)
        
        checkpoint_path = f"/kaggle/working/checkpoint_ep{epoch+1}.pth"
        base_deno = denoiser_core.module if hasattr(denoiser_core, 'module') else denoiser_core
        base_stride = audio_strider.module if hasattr(audio_strider, 'module') else audio_strider
        
        checkpoint = {
            'epoch': epoch + 1,
            'output_mode': OUTPUT_MODE.value,
            'denoiser_type': DENOISER.value,
            'audio_strider_state': base_stride.state_dict(),
            'denoiser_state': base_deno.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'ema_audio_strider_state': ema_audio_strider.module.state_dict() if hasattr(ema_audio_strider, 'module') else ema_audio_strider.state_dict(),
            'ema_denoiser_state': ema_denoiser.module.state_dict() if hasattr(ema_denoiser, 'module') else ema_denoiser.state_dict(),
            'best_val_loss': best_val_loss
        }
        
        torch.save(checkpoint, checkpoint_path)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(checkpoint, "/kaggle/working/best_model.pth")
            print(f"| {epoch + 1:^7} | {avg_epoch_loss:^15.4f} | {avg_val_loss:^15.4f} | --> Saved: {checkpoint_path} & BEST")
        else:
            print(f"| {epoch + 1:^7} | {avg_epoch_loss:^15.4f} | {avg_val_loss:^15.4f} | --> Saved: {checkpoint_path}")

    if time.time() - start_time > max_time_seconds:
        break

In [ ]:
@torch.no_grad()
def streaming_inference(diffusion_model, audio_strider, long_audio_tensor, timesteps=50, chunk_size_50hz=150, cfg_scale=1.7):
    diffusion_model.eval()
    audio_strider.eval()
    
    target_dim = 53 if OUTPUT_MODE == OutputMode.FLAME else 32
    
    total_time = long_audio_tensor.shape[2]
    hidden_state = None
    all_predicted_outputs = []
    
    t_steps = torch.linspace(0.0, 1.0, timesteps + 1, device=device)
    
    for start_idx in range(0, total_time, chunk_size_50hz):
        end_idx = min(start_idx + chunk_size_50hz, total_time)
        audio_chunk = long_audio_tensor[:, :, start_idx:end_idx]
        
        strided_audio = audio_strider(audio_chunk)
        uncond_audio = torch.zeros_like(strided_audio)
        
        xt = torch.randn(1, strided_audio.shape[1], target_dim, device=device)
        
        for i in range(timesteps):
            t = t_steps[i].unsqueeze(0).expand(xt.shape[0])
            t_next = t_steps[i + 1].unsqueeze(0).expand(xt.shape[0])
            
            v_t_cond, current_hidden_state = diffusion_model.model(xt, strided_audio, t, hidden_state)
            v_t_uncond, _ = diffusion_model.model(xt, uncond_audio, t, hidden_state)
            
            v_t = v_t_uncond + cfg_scale * (v_t_cond - v_t_uncond)
            
            dt = t_next - t
            xt = xt + v_t * dt.view(-1, 1, 1)
            
        hidden_state = current_hidden_state.detach() if current_hidden_state is not None else None
        all_predicted_outputs.append(xt)
        
    full_output = torch.cat(all_predicted_outputs, dim=1)
    
    if OUTPUT_MODE == OutputMode.LATENT:
        full_output = vae_model.decoder(full_output)
        
    return full_output

In [ ]:
def export_prediction_to_video(predicted_tensor, video_filename, audio_path, flame_model, device, sample_flame_path, fps=25):
    if hasattr(flame_model, 'module'):
        flame_model = flame_model.module

    if predicted_tensor.dim() == 3:
        predicted_tensor = predicted_tensor.squeeze(0)
        
    audio_clip = AudioFileClip(audio_path)
    audio_duration = audio_clip.duration
    target_frame_count = int(np.ceil(audio_duration * fps))

    def adjust_tensor(tensor):
        current_frames = tensor.shape[0]
        if current_frames < target_frame_count:
            padding = tensor[-1:].expand(target_frame_count - current_frames, -1)
            tensor = torch.cat([tensor, padding], dim=0)
        elif current_frames > target_frame_count:
            tensor = tensor[:target_frame_count]
        return tensor

    if predicted_tensor.shape[0] < 5:
        return

    predicted_tensor = adjust_tensor(predicted_tensor)
    T = target_frame_count
    
    gt_dict = torch.load(sample_flame_path, weights_only=True, map_location=device)
    base_shape = gt_dict['shape'][0:1].to(device)
    
    expressions = gt_dict['exp']
    jaw_pose = gt_dict['pose'][:, 3:6]
    gt_tensor = torch.cat([expressions, jaw_pose], dim=-1)[:, :53]
    gt_tensor = adjust_tensor(gt_tensor)

    def get_verts(tensor):
        all_verts = []
        chunk_size = 500
        with torch.no_grad():
            for i in range(0, T, chunk_size):
                chunk = tensor[i:i+chunk_size].to(device)
                fixed_shape = base_shape.expand(chunk.shape[0], -1)
                pose = torch.zeros((chunk.shape[0], 6), device=chunk.device, dtype=chunk.dtype)
                pose[:, 3:6] = chunk[:, 50:53]
                flame_output = flame_model(shape_params=fixed_shape, expression_params=chunk[:, :50], pose_params=pose)
                all_verts.append(flame_output[0].cpu().numpy())
        return np.concatenate(all_verts, axis=0)

    pred_verts = get_verts(predicted_tensor)
    gt_verts = get_verts(gt_tensor)
    faces = flame_model.faces_tensor.cpu().numpy()

    renderer = pyrender.OffscreenRenderer(viewport_width=512, viewport_height=512)
    camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0, aspectRatio=1.0)
    camera_pose = np.array([
        [1.0,  0.0,  0.0,  0.0],
        [0.0,  1.0,  0.0,  0.0],
        [0.0,  0.0,  1.0,  0.3], 
        [0.0,  0.0,  0.0,  1.0]
    ])
    
    light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=2.0)
    temp_video = "temp_silent.mp4"
    writer = imageio.get_writer(temp_video, fps=fps)
    
    for i in range(T):
        gt_mesh = trimesh.Trimesh(vertices=gt_verts[i], faces=faces)
        gt_mat = pyrender.MetallicRoughnessMaterial(metallicFactor=0.1, alphaMode='OPAQUE', baseColorFactor=(0.6, 0.7, 0.6, 1.0))
        scene_gt = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0])
        scene_gt.add(pyrender.Mesh.from_trimesh(gt_mesh, material=gt_mat))
        scene_gt.add(camera, pose=camera_pose)
        scene_gt.add(light, pose=camera_pose)
        color_gt, _ = renderer.render(scene_gt)

        pred_mesh = trimesh.Trimesh(vertices=pred_verts[i], faces=faces)
        pred_mat = pyrender.MetallicRoughnessMaterial(metallicFactor=0.1, alphaMode='OPAQUE', baseColorFactor=(0.7, 0.6, 0.6, 1.0))
        scene_pred = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0])
        scene_pred.add(pyrender.Mesh.from_trimesh(pred_mesh, material=pred_mat))
        scene_pred.add(camera, pose=camera_pose)
        scene_pred.add(light, pose=camera_pose)
        color_pred, _ = renderer.render(scene_pred)
        
        color_concat = np.concatenate((color_gt, color_pred), axis=1)
        writer.append_data(color_concat)
        
    writer.close()
    renderer.delete()
    
    video_clip = VideoFileClip(temp_video)
    final_clip = video_clip.set_audio(audio_clip)
    final_clip.write_videofile(video_filename, codec="libx264", audio_codec="aac", fps=fps, verbose=False, logger=None)
    os.remove(temp_video)

In [ ]:
import time

def generate_inference_videos(id_list, lang, diffusion_model, audio_strider, flame_model, device):
    base_flame_model = flame_model.module if hasattr(flame_model, 'module') else flame_model

    for item_id in id_list:
        sample_wav_path = f"/kaggle/input/datasets/jorx04/fully-collected-dataset/audio/{lang}/{item_id}.wav"
        sample_audio_path = f"/kaggle/input/datasets/jorx04/mhubert-147-features-{lang}/{lang}/{item_id}.pt"
        sample_flame_path = f"/kaggle/input/datasets/jorx04/fully-collected-dataset/flame/{lang}/{item_id}_flame_params.pt"
        output_vid = f"/kaggle/working/vid_recon_{item_id}.mp4"
        
        if os.path.exists(sample_audio_path) and os.path.exists(sample_wav_path) and os.path.exists(sample_flame_path):
            start_time = time.time()
            
            raw_audio_feat = torch.load(sample_audio_path, weights_only=True)
            long_audio_tensor = raw_audio_feat.unsqueeze(0).to(device)
        
            predicted_sequence = streaming_inference(
                diffusion_model=diffusion_model, 
                audio_strider=audio_strider, 
                long_audio_tensor=long_audio_tensor, 
                timesteps=INFER_TIMESTEPS, 
                chunk_size_50hz=200,
                cfg_scale=1.7
            )
            
            export_prediction_to_video(
                predicted_tensor=predicted_sequence,
                video_filename=output_vid,
                audio_path=sample_wav_path,
                flame_model=base_flame_model,
                device=device,
                sample_flame_path=sample_flame_path
            )
            
            elapsed = time.time() - start_time
            print(f"Video {item_id} generated in {elapsed:.2f} seconds")

In [ ]:
ara_files = []
eng_files = []

val_dataset.dataset.max_len = None 

for i in range(len(val_dataset)):
    if len(ara_files) >= 3 and len(eng_files) >= 3:
        break

    orig_idx = val_dataset.indices[i]
    
    h_path, t_path = val_dataset.dataset.samples[orig_idx]
    
    is_ara = '/ara/' in t_path
    is_eng = '/eng/' in t_path
    
    if is_ara and len(ara_files) >= 3:
        continue
    if is_eng and len(eng_files) >= 3:
        continue

    audio_seq, flame_seq, shape_param = val_dataset[i]
    
    if flame_seq.shape[0] >= 150:
        base_name = os.path.basename(t_path).replace("_flame_params.pt", "").replace(".pt", "")
        if is_ara:
            ara_files.append(base_name)
        elif is_eng:
            eng_files.append(base_name)

print("Arabic Files (>= 150 frames):", ara_files)
print("English Files (>= 150 frames):", eng_files)

val_dataset.dataset.max_len = 75

In [ ]:
ara_files.extend(["0YJdaGwpUHA_0", "0YJdaGwpUHA_1"])
eng_files.extend(["-WB_SGqeJo8_15", "-m8tHXm8D8w_18"])

In [ ]:
current_epoch = epoch if 'epoch' in globals() else 0

eval_diffusion = ema_diffusion_model if current_epoch >= EMA_START_EPOCH else diffusion_model
eval_strider = ema_audio_strider if current_epoch >= EMA_START_EPOCH else audio_strider

generate_inference_videos(ara_files, "ara", eval_diffusion, eval_strider, flame_model, device)
generate_inference_videos(eng_files, "eng", eval_diffusion, eval_strider, flame_model, device)

In [ ]:
!rm -r /kaggle/working/spectre
!rm -r /kaggle/working/latents